SETUP

In [15]:
# If Colab asks to restart after installing, accept it.
!pip -q install --upgrade pip
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.0" "sentence-transformers>=3.0.1" \
                 "faiss-cpu>=1.8.0" "langchain-text-splitters>=0.3.0" "pypdf>=4.2.0" \
                 "gradio>=4.44.0"
# bitsandbytes is optional (for 8-bit loading if a GPU is available). It may fail on CPU-only.
!pip -q install bitsandbytes==0.44.1 || echo "bitsandbytes optional install skipped"
!pip -q install -U google-generativeai
!pip install pypdf python-docx

In [16]:
import os, torch, faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch: 2.10.0+cu128 | CUDA available: True


In [17]:
import os
import getpass

# 1) Try existing env first
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "").strip()

# 2) Optional: try Colab Secret if available
if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = (userdata.get("GOOGLE_API_KEY") or "").strip()
    except Exception:
        GOOGLE_API_KEY = ""

# 3) Manual input fallback (hidden)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass.getpass("Enter GOOGLE_API_KEY: ").strip()
    # <GOOGLE_API_KEY_REMOVED>


if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("Gemini key loaded: ✅")
else:
    print("Gemini key loaded: ❌ Missing")


Gemini key loaded: ✅


CONFIGURE MODEL BACKEND

In [18]:
# === Choose your generator backend ===
# "local" uses a small open-source chat model via Hugging Face Transformers.
# "openai" uses OpenAI's API (set OPENAI_API_KEY).
# "gemini" uses Google Generative AI (set GOOGLE_API_KEY).
GEN_BACKEND = "gemini"  # options: "local", "openai", "gemini"

# Local model config:
LOCAL_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # light model for CPU/GPU demos

# Retrieval config
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 4

# Prompt template
SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the provided context to answer the user's question.\n"
    "If the answer is not in the context, say you don't know.\n"
)

ANSWER_TEMPLATE = """[System]
{system}

[Context]
{context}

[User Question]
{question}

[Instructions]
- Cite the most relevant chunks briefly (e.g., 'From chunk 2').
- If unsure, say 'I don't know from the provided docs.'
- Keep answers concise and factual.
"""

OPENAI_API_KEY = globals().get("OPENAI_API_KEY", os.getenv("OPENAI_API_KEY", ""))
GOOGLE_API_KEY = globals().get("GOOGLE_API_KEY", os.getenv("GOOGLE_API_KEY", ""))

LOAD FILES/CHUNKS AND EMBEDDED

In [19]:
import os
from typing import List, Dict
from pypdf import PdfReader
from docx import Document  # ADDED THIS LINE

def load_texts_from_paths(paths: List[str]) -> List[Dict]:
    """
    Load text from various file formats: PDF, DOCX, DOC, TXT, MD
    """
    docs = []
    for p in paths:
        text = ""

        # PDF files
        if p.lower().endswith(".pdf"):
            try:
                reader = PdfReader(p)
                for page in reader.pages:
                    text += page.extract_text() or ""
                print(f"✓ Loaded PDF: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse PDF {p}: {e}")
                continue

        # DOCX files (Word documents) # NEW
        elif p.lower().endswith(".docx"):
            try:
                doc = Document(p)
                # Extract text from all paragraphs
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
                # Also extract text from tables
                for table in doc.tables:
                    for row in table.rows:
                        for cell in row.cells:
                            text += "\n" + cell.text
                print(f"✓ Loaded DOCX: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse DOCX {p}: {e}")
                continue

        # DOC files (legacy Word format) # NEW
        elif p.lower().endswith(".doc"):
            try:
                doc = Document(p)
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
                print(f"✓ Loaded DOC: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse DOC {p}: {e}")
                print("     Note: Legacy .doc files may require conversion to .docx")
                continue

        # TXT and MD files
        elif p.lower().endswith((".txt", ".md")):
            try:
                with open(p, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                print(f"✓ Loaded TXT/MD: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to read text file {p}: {e}")
                continue

        else:
            print(f"[SKIP] Unsupported file type: {p}")
            continue

        if text.strip():  # Only add if we got some text
            docs.append({"path": p, "text": text})
        else:
            print(f"[WARN] No text extracted from {p}")

    return docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
)

def chunk_docs(docs: List[Dict]) -> List[Dict]:
    chunks = []
    for d in docs:
        for i, ch in enumerate(splitter.split_text(d["text"])):
            chunks.append({"source": d["path"], "chunk_id": i, "text": ch})
    return chunks

class RAGIndex:
    def __init__(self, embedding_model_name: str):
        self.model = SentenceTransformer(embedding_model_name, device=device)
        self.index = None
        self.chunks: List[Dict] = []

    def build(self, chunks: List[Dict]):
        self.chunks = chunks
        embs = self.model.encode([c["text"] for c in chunks], convert_to_numpy=True, show_progress_bar=True)
        dim = embs.shape[1]
        index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(embs)
        index.add(embs)
        self.index = index
        print(f"Built index with {len(chunks)} chunks.")

    def search(self, query: str, k: int = 4):
        if self.index is None or not self.chunks:
            return []
        q = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q)
        scores, idxs = self.index.search(q, k)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1: continue
            results.append((float(score), self.chunks[idx]))
        return results

rag = RAGIndex(EMBEDDING_MODEL)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GENERATIONS BACKEND

In [20]:
def render_context(snippets):
    lines = []
    for rank, (score, ch) in enumerate(snippets, start=1):
        header = f"[Chunk {rank}] (score={score:.3f}) source={os.path.basename(ch['source'])} id={ch['chunk_id']}"
        lines.append(header + "\n" + ch["text"])
    return "\n\n".join(lines)

def build_prompt(question, context_blocks):
    return ANSWER_TEMPLATE.format(
        system=SYSTEM_PROMPT.strip(),
        context=context_blocks.strip(),
        question=question.strip()
    )

In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

_local_pipe = None

def get_local_pipe():
    global _local_pipe
    if _local_pipe is None:
        tok = AutoTokenizer.from_pretrained(LOCAL_MODEL, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            low_cpu_mem_usage=True
        )
        _local_pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tok,
            device=0 if torch.cuda.is_available() else -1,
            max_new_tokens=384,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.05
        )
    return _local_pipe

def generate_local(prompt: str) -> str:
    p = get_local_pipe()
    out = p(prompt, pad_token_id=p.tokenizer.eos_token_id)[0]["generated_text"]
    return out[len(prompt):].strip()

In [22]:
def generate_openai(prompt: str) -> str:
    if not OPENAI_API_KEY:
        return "OPENAI_API_KEY not set. Switch GEN_BACKEND to 'local' or set your key."
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"system","content":SYSTEM_PROMPT},
                      {"role":"user","content":prompt}],
            temperature=0.2,
            max_tokens=400
        )
        return r.choices[0].message.content
    except Exception as e:
        return f"[OpenAI error] {e}"

In [23]:
def _strip_local_proxy_env():
    # VS Code Colab plugin may inject localhost proxy envs that timeout.
    keys = [
        "HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",
        "http_proxy", "https_proxy", "all_proxy",
    ]
    for k in keys:
        v = os.environ.get(k, "")
        if "localhost:" in v or "127.0.0.1:" in v:
            os.environ.pop(k, None)


def _extract_gemini_text(resp_json: dict) -> str:
    cands = resp_json.get("candidates", [])
    if not cands:
        return ""
    parts = cands[0].get("content", {}).get("parts", [])
    texts = [p.get("text", "") for p in parts if isinstance(p, dict)]
    return "\n".join([t for t in texts if t]).strip()


def _generate_gemini_via_sdk(prompt: str, model_name: str = "gemini-2.5-flash-lite", timeout_sec: int = 35) -> str:
    import google.generativeai as genai

    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel(model_name)
    r = model.generate_content(prompt, request_options={"timeout": timeout_sec})
    return (getattr(r, "text", "") or "").strip()


def _generate_gemini_via_rest(prompt: str, model_name: str = "gemini-2.5-flash-lite", timeout_sec: int = 35) -> str:
    import requests

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}

    # Explicitly disable proxies for this request.
    r = requests.post(
        url,
        params={"key": GOOGLE_API_KEY},
        json=payload,
        timeout=timeout_sec,
        proxies={"http": "", "https": ""},
    )
    r.raise_for_status()
    data = r.json()
    text = _extract_gemini_text(data)
    if text:
        return text
    return f"[Gemini REST response without text] {str(data)[:400]}"


def generate_gemini(prompt: str) -> str:
    if not GOOGLE_API_KEY:
        return "GOOGLE_API_KEY not set. Switch GEN_BACKEND to 'local' or set your key."

    _strip_local_proxy_env()

    try:
        return _generate_gemini_via_sdk(prompt)
    except Exception as sdk_err:
        try:
            return _generate_gemini_via_rest(prompt)
        except Exception as rest_err:
            return f"[Gemini error] sdk={sdk_err} | rest={rest_err}"


ASK QUESTIONS

In [24]:
def answer_question(question: str, top_k: int = TOP_K):
    hits = rag.search(question, k=top_k)
    context = render_context(hits)
    prompt = build_prompt(question, context)

    if GEN_BACKEND == "local":
        answer = generate_local(prompt)
    elif GEN_BACKEND == "openai":
        answer = generate_openai(prompt)
    elif GEN_BACKEND == "gemini":
        answer = generate_gemini(prompt)
    else:
        answer = f"Unknown backend: {GEN_BACKEND}"

    return {"question": question, "answer": answer, "top_chunks": hits}

print("RAG ready. After indexing, call: answer_question('Your query')")

RAG ready. After indexing, call: answer_question('Your query')


In [25]:
import os

print("GOOGLE_API_KEY loaded:", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")
print("Connectivity test:")
print(generate_gemini("Say only: ready"))


GOOGLE_API_KEY loaded: ✅
Connectivity test:
ready


In [26]:
import time

start = time.time()
resp = generate_gemini("Say only: ready")
elapsed = round(time.time() - start, 2)
print(resp, "⏱", elapsed, "s")


ready ⏱ 35.59 s


UPLOAD FILESAND BUILD INDEX

In [27]:
import os
from typing import List

SUPPORTED_INDEX_EXTS = (".pdf", ".txt", ".md", ".docx", ".doc")


def _collect_supported_files(scan_dir: str) -> List[str]:
    paths = []
    if not scan_dir or not os.path.isdir(scan_dir):
        return paths

    for root, _, files in os.walk(scan_dir):
        for name in files:
            if name.lower().endswith(SUPPORTED_INDEX_EXTS):
                paths.append(os.path.join(root, name))
    return sorted(paths)


def _upload_files_widget() -> List[str]:
    """Works only in active Chrome Colab browser sessions."""
    try:
        from google.colab import files
    except Exception as e:
        print(f"Colab upload widget unavailable: {e}")
        return []

    try:
        print("Select PDFs/TXT/MD/DOCX/DOC files...")
        uploaded = files.upload()
    except Exception as e:
        print(f"Upload widget failed: {e}")
        return []

    paths = []
    target_dir = "/content" if os.path.isdir("/content") else os.getcwd()
    for name, data in uploaded.items():
        out_path = os.path.join(target_dir, name)
        with open(out_path, "wb") as f:
            f.write(data)
        paths.append(out_path)
    return paths


def resolve_index_paths(
    manual_paths: List[str] = None,
    scan_dir: str = "",
    use_widget_upload: bool = False,
) -> List[str]:
    # 1) Highest priority: explicit manual paths
    manual_paths = manual_paths or []
    existing = [p for p in manual_paths if isinstance(p, str) and os.path.isfile(p)]
    if existing:
        return existing

    # 2) Optional browser widget upload (Chrome Colab only)
    if use_widget_upload:
        widget_paths = _upload_files_widget()
        if widget_paths:
            return widget_paths
        print("Widget upload returned no files. Falling back to directory scan...")

    # 3) Directory scan fallback
    scanned = _collect_supported_files(scan_dir)
    if scanned:
        print(f"Found {len(scanned)} supported files in: {scan_dir}")
    return scanned


def build_index_from_paths(paths: List[str]):
    if not paths:
        raise ValueError("No input files found. Set MANUAL_PATHS or SCAN_DIR first.")

    docs = load_texts_from_paths(paths)
    chunks = chunk_docs(docs)
    rag.build(chunks)
    print(f"Indexed {len(chunks)} chunks from {len(docs)} files.")


In [28]:
# ===== Build index in both VS Code and Chrome Colab =====
# VS Code / non-browser Colab: keep USE_WIDGET_UPLOAD = False, then use MANUAL_PATHS or SCAN_DIR.
# Chrome Colab page: set USE_WIDGET_UPLOAD = True to open file picker.

USE_WIDGET_UPLOAD = False

# Option A: explicit files (recommended in VS Code)
MANUAL_PATHS = [
    # "/content/drive/MyDrive/your_script.pdf",
]

# Option B: auto-scan folder if MANUAL_PATHS is empty
if os.path.isdir("/content/drive/MyDrive"):
    SCAN_DIR = "/content/drive/MyDrive"
else:
    SCAN_DIR = os.getcwd()

paths = resolve_index_paths(
    manual_paths=MANUAL_PATHS,
    scan_dir=SCAN_DIR,
    use_widget_upload=USE_WIDGET_UPLOAD,
)

if not paths:
    raise ValueError(
        "No files found for indexing. Add paths to MANUAL_PATHS, or change SCAN_DIR, "
        "or set USE_WIDGET_UPLOAD=True in Chrome Colab."
    )

build_index_from_paths(paths)

# Sanity checks
print("Files:", paths[:5], "..." if len(paths) > 5 else "")
print("Chunks indexed:", len(rag.chunks))
print("First chunk preview:\n", rag.chunks[0]["text"][:500] if rag.chunks else "No chunks")


Found 1 supported files in: /content
✓ Loaded TXT/MD: README.md (962 chars)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Built index with 2 chunks.
Indexed 2 chunks from 1 files.
Files: ['/content/sample_data/README.md'] 
Chunks indexed: 2
First chunk preview:
 This directory includes a few sample datasets to get you started.

*   `california_housing_data*.csv` is California housing data from the 1990 US
    Census; more information is available at:
    https://docs.google.com/document/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub

*   `mnist_*.csv` is a small sample of the
    [MNIST database](https://en.wikipedia.org/wiki/MNIST_database), which is
    described at: http://yann.lecun.com/exdb/mnist/

* 


GRADIO CHAT OVERLAY

In [29]:
import os
import gradio as gr
from pypdf import PdfReader
from docx import Document


def _extract_uploaded_path(uploaded_file) -> str:
    if uploaded_file is None:
        return ""
    if isinstance(uploaded_file, str):
        return uploaded_file
    if isinstance(uploaded_file, list) and uploaded_file:
        return _extract_uploaded_path(uploaded_file[0])
    if isinstance(uploaded_file, dict):
        return uploaded_file.get("name", "") or uploaded_file.get("path", "")
    if hasattr(uploaded_file, "name"):
        return uploaded_file.name
    return ""


def read_file_to_string(file_path: str) -> str:
    """Reads a TXT, PDF, DOCX, or DOC file into a single text string."""
    if not file_path:
        return ""

    text = ""
    file_ext = os.path.splitext(file_path)[1].lower()

    try:
        if file_ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                text += page.extract_text() or ""

        elif file_ext == ".docx":
            doc = Document(file_path)
            text = "\n".join([p.text for p in doc.paragraphs])
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        text += "\n" + cell.text

        elif file_ext == ".doc":
            try:
                doc = Document(file_path)
                text = "\n".join([p.text for p in doc.paragraphs])
            except Exception:
                return "ERROR: Legacy .doc detected. Please resave as .docx and upload again."

        elif file_ext in [".txt", ".md"]:
            with open(file_path, "r", encoding="utf-8", errors="replace") as f:
                text = f.read()

        else:
            return f"ERROR: Unsupported file type: {file_ext}"

    except Exception as e:
        return f"ERROR: Could not read file ({e})"

    return text


def on_upload(uploaded_file):
    file_path = _extract_uploaded_path(uploaded_file)
    if not file_path:
        return "", "No file uploaded."

    file_text = read_file_to_string(file_path)
    if file_text.startswith("ERROR:"):
        return "", file_text

    filename = os.path.basename(file_path)
    return file_text, f"Uploaded file: {filename}"


def clear_uploaded_state():
    return "", "No file uploaded."


def _combine_user_input(story_text: str, uploaded_text: str) -> str:
    story = (story_text or "").strip()
    file_body = (uploaded_text or "").strip()

    if story and file_body:
        return f"{story}\n\n[Uploaded File Content]\n{file_body}"
    return story or file_body


def _gradio_story_eval(story_text: str, mission: str = ""):
    """Runs an Impact-Studios-style evaluation grounded in the RAG index."""
    if not story_text.strip():
        return "Please enter text or upload a file before submitting."

    base_prompt = """
    You are an editorial advisor focused on community engagement and ethical publication. Read the following piece of writing
    and identify the communities, audiences, or individuals who may be directly affected by its themes. Then provide specific,
    actionable suggestions for how the author can responsibly and meaningfully reach out to or support those communities.

    Your feedback must:
    - Be concrete (exact additions, placements, wording ideas, or resources).
    - Explain why each suggestion is appropriate for the content.
    - Address ethical considerations such as harm reduction, accessibility, and care for vulnerable readers when relevant.

    Avoid generic advice like "be sensitive" or "raise awareness."

    When users share a script, concept, or idea, your job is to:
    1. Analyze the submission's tone, themes, and emotional depth.
    2. Determine the topics discussed in the submission.
    3. Provide clear reasoning.
    4. Provide specific and actionable outreach/support suggestions.
    5. Integrate any mission context naturally.

    Respond in this format:

    Topics identified:
    (List topics found in the submission. For each topic include one quote and one actionable recommendation.)
    """

    if mission and mission.strip():
        user_prompt = (
            f"{base_prompt}\n\nCurrent Studio Mission:\n{mission.strip()}\n\nUser Submission:\n{story_text.strip()}"
        )
    else:
        user_prompt = f"{base_prompt}\n\nUser Submission:\n{story_text.strip()}"

    context_text = ""
    if "rag" in globals() and getattr(rag, "chunks", []):
        hits = rag.search(story_text, k=TOP_K)
        context_text = render_context(hits)
    else:
        hits = []

    full_prompt = ANSWER_TEMPLATE.format(
        system=SYSTEM_PROMPT,
        context=context_text or "[No extra context]",
        question=user_prompt,
    )

    out = answer_question(full_prompt)
    answer = out.get("answer", "")

    cites = []
    if context_text:
        for rank, (_, ch) in enumerate(out.get("top_chunks", []), start=1):
            cites.append(f"Chunk {rank} - {os.path.basename(ch['source'])}#{ch['chunk_id']}")
    suffix = ("\n\n---\nSources: " + "; ".join(cites)) if cites else ""

    return (answer or "[Empty answer]") + suffix


def handle_evaluation(story_text: str, mission_text: str, uploaded_text: str, chat_history):
    merged_submission = _combine_user_input(story_text, uploaded_text)
    if not merged_submission.strip():
        return chat_history, ""

    evaluation = _gradio_story_eval(merged_submission, mission_text or "")

    story_display = (story_text or "").strip()
    if story_display and (uploaded_text or "").strip():
        user_turn = story_display + "\n[Attachment included]"
    elif story_display:
        user_turn = story_display
    else:
        user_turn = "[Attachment only submission]"

    chat_history = chat_history or []
    chat_history.append((user_turn, evaluation))

    return chat_history, ""


CUSTOM_CSS = """
.gradio-container {
    background: linear-gradient(135deg, #E0F2F7 0%, #B3E5FC 100%);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
}

#main-layout {
    min-height: 100vh;
    margin: 0;
    width: 100%;
}

.sidebar {
    background: #E8F5F9;
    border-right: 1px solid #B3E5FC;
    padding: 16px;
    min-height: 100vh;
}

.logo {
    font-size: 16px;
    font-weight: 600;
    color: #01579B;
    background: #FFFFFF;
    padding: 8px 12px;
    border-radius: 8px;
    margin-bottom: 16px;
    display: inline-block;
}

.sidebar-button button {
    width: 100%;
    text-align: left;
    border-radius: 8px;
    border: 1px solid #81D4FA;
    color: #0277BD;
    background: #FFFFFF;
}

.sidebar-button button:hover {
    background: #E1F5FE;
    border-color: #4FC3F7;
}

.main-content {
    background: #FFFFFF;
    min-height: 100vh;
    display: flex;
    flex-direction: column;
}

.header {
    border-bottom: 1px solid #E0E0E0;
    padding: 16px 20px;
}

.header-title {
    margin: 0;
    color: #01579B;
    font-size: 24px;
}

.chat-area {
    flex: 1;
    overflow-y: auto;
    padding: 20px;
    background: #E1F5FE;
}

.input-container {
    border-top: 1px solid #E0E0E0;
    padding: 12px 16px;
    background: #FFFFFF;
}

.input-row {
    display: flex;
    align-items: center;
    gap: 10px;
    border: 1px solid #E0E0E0;
    border-radius: 18px;
    padding: 8px 10px;
}

.upload-button button {
    width: 40px;
    min-width: 40px;
    height: 40px;
    border-radius: 50%;
    border: 1px solid #81D4FA;
    background: #FFFFFF;
    color: #0277BD;
    font-size: 18px;
    padding: 0;
}

.text-input textarea {
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
}

.submit-button button {
    width: 40px;
    min-width: 40px;
    height: 40px;
    border-radius: 50%;
    border: none;
    background: #01579B;
    color: #FFFFFF;
    font-weight: 700;
}

.submit-button button:hover {
    background: #0277BD;
}

.file-status {
    color: #546E7A;
    font-size: 12px;
    margin-top: 8px;
    word-break: break-word;
}

.file-status p {
    margin: 0;
}

.clear-file-button button {
    margin-top: 8px;
    border: 1px solid #B0BEC5;
    color: #455A64;
    background: #FFFFFF;
    border-radius: 6px;
}

@media (max-width: 768px) {
    .sidebar {
        min-height: auto;
    }
}
"""


with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), fill_height=True) as demo:
    uploaded_text_state = gr.State("")

    with gr.Row(elem_id="main-layout"):
        with gr.Column(scale=0, min_width=260, elem_classes="sidebar"):
            gr.HTML('<div class="logo">Impact Studios</div>')

            new_chat_btn = gr.Button("New chat", elem_classes="sidebar-button")
            search_btn = gr.Button("Search chat", elem_classes="sidebar-button")
            add_agent_btn = gr.Button("Add agent", elem_classes="sidebar-button")

            mission_box = gr.Textbox(
                label="Studio Mission (optional)",
                placeholder="Add mission context for this run...",
                lines=6,
            )

        with gr.Column(scale=1, elem_classes="main-content"):
            gr.HTML(
                """
                <div class="header">
                    <h1 class="header-title">Impact Studios</h1>
                </div>
                """
            )

            with gr.Column(scale=1, elem_classes="chat-area"):
                chatbot = gr.Chatbot(
                    label="",
                    show_label=False,
                    container=False,
                    bubble_full_width=False,
                    height=None,
                )

            with gr.Column(elem_classes="input-container"):
                with gr.Row(elem_classes="input-row", equal_height=True):
                    upload_btn = gr.UploadButton(
                        "Attach",
                        file_types=[".txt", ".pdf", ".docx", ".doc", ".md"],
                        file_count="single",
                        elem_classes="upload-button",
                        scale=0,
                    )

                    story = gr.Textbox(
                        label="",
                        placeholder="Ask anything",
                        lines=1,
                        max_lines=4,
                        show_label=False,
                        container=False,
                        elem_classes="text-input",
                        scale=1,
                    )

                    submit_btn = gr.Button("^", elem_classes="submit-button", scale=0)

                file_status = gr.Markdown("No file uploaded.", elem_classes="file-status")
                clear_file_btn = gr.Button("Clear attached file", elem_classes="clear-file-button")

    upload_btn.upload(
        fn=on_upload,
        inputs=upload_btn,
        outputs=[uploaded_text_state, file_status],
    )

    clear_file_btn.click(
        fn=clear_uploaded_state,
        inputs=None,
        outputs=[uploaded_text_state, file_status],
    )

    submit_btn.click(
        fn=handle_evaluation,
        inputs=[story, mission_box, uploaded_text_state, chatbot],
        outputs=[chatbot, story],
    )

    story.submit(
        fn=handle_evaluation,
        inputs=[story, mission_box, uploaded_text_state, chatbot],
        outputs=[chatbot, story],
    )

    def clear_chat():
        return [], ""

    new_chat_btn.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chatbot, story],
    )

print("Launching Impact Studios Evaluator...")
demo.launch(share=True, debug=True)


/tmp/ipython-input-3696081189.py:316: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), fill_height=True) as demo:
/tmp/ipython-input-3696081189.py:316: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(), fill_height=True) as demo:
/tmp/ipython-input-3696081189.py:343: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipython-input-3696081189.py:343: DeprecationWarning: The 'bubble_full_widt

Launching Impact Studios Evaluator...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://11b1c46487bbac7bf4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://11b1c46487bbac7bf4.gradio.live


In [ ]:
#Use this if first code block fails
import os, gradio as gr

def _gradio_ask(q: str):
    if not q.strip():
        return "Please enter a question."
    print("Using backend:", GEN_BACKEND)  # shows in Colab logs
    out = answer_question(q)
    cites = []
    for rank, (_, ch) in enumerate(out["top_chunks"], start=1):
        cites.append(f"Chunk {rank} — {os.path.basename(ch['source'])}#{ch['chunk_id']}")
    suffix = ("\n\n---\nSources: " + "; ".join(cites)) if cites else ""
    return (out["answer"] or "[Empty answer]") + suffix

with gr.Blocks() as demo:
    gr.Markdown("### RAG Chat — ask questions grounded in your uploaded docs")
    q = gr.Textbox(label="Your question")
    a = gr.Markdown(label="Answer")
    btn = gr.Button("Ask")
    btn.click(_gradio_ask, inputs=q, outputs=a)

print("Launching RAG chat…")
demo.launch(share=True, debug=True)